# S4_4 — Linear Probe Monitoring Analysis

**Leaf-JEPA IRP** | Stage 4 — Leaf JEPA Pretraining


Loads pretraining history and produces detailed convergence analysis.
This notebook produces the dissertation figures for:
- Pretraining loss convergence
- Representation quality curve (LP monitor)
- EMA schedule plot
- Convergence detection (when did learning plateau?)


## Initialization

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import set_seed

set_seed(RANDOM_SEED)

history_path    = PRETRAIN_DIR / "pretrain_history.json"
lp_history_path = PRETRAIN_DIR / "lp_monitor_history.json"

assert history_path.exists(),    f"Run S4_3 first: {history_path}"
assert lp_history_path.exists(), f"Run S4_3 first: {lp_history_path}"

with open(history_path)    as f: history    = json.load(f)
with open(lp_history_path) as f: lp_history = json.load(f)

epochs    = [h["epoch"] for h in history]
losses    = [h["loss"]  for h in history]
taus      = [h["tau"]   for h in history]
lrs       = [h["lr"]    for h in history]
lp_epochs = [h["pretrain_epoch"]   for h in lp_history]
lp_f1s    = [h["lp_val_macro_f1"]  for h in lp_history]

print(f"History loaded: {len(history)} epochs")
print(f"LP monitor:     {len(lp_history)} checkpoints")
if lp_history:
    best = max(lp_history, key=lambda r: r["lp_val_macro_f1"])
    print(f"Best LP F1: {best['lp_val_macro_f1']:.4f} at epoch {best['pretrain_epoch']}")


History loaded: 125 epochs
LP monitor:     5 checkpoints
Best LP F1: 0.8848 at epoch 25


## Training Curve

In [2]:
window = 10
rolling = [np.mean(losses[max(0,i-window+1):i+1]) for i in range(len(losses))]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Loss with rolling average
axes[0,0].plot(epochs, losses,   color="#4fc3f7", alpha=0.4, lw=1, label="Batch loss")
axes[0,0].plot(epochs, rolling,  color="#0288d1",             lw=2, label=f"{window}-epoch rolling avg")
if lp_history:
    best = max(lp_history, key=lambda r: r["lp_val_macro_f1"])
    axes[0,0].axvline(best["pretrain_epoch"], color="#ef5350", lw=1.5, ls="--",
                       label=f"Best LP epoch ({best['pretrain_epoch']})")
axes[0,0].set_xlabel("Epoch"); axes[0,0].set_ylabel("Pretraining Loss")
axes[0,0].set_title("Pretraining Loss Convergence"); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# LP F1 quality curve
if lp_history:
    axes[0,1].plot(lp_epochs, lp_f1s, color="#ffb74d", lw=2, marker="o", ms=7)
    axes[0,1].fill_between(lp_epochs, 0, lp_f1s, alpha=0.15, color="#ffb74d")
    axes[0,1].axhline(max(lp_f1s), color="#ef5350", ls="--", lw=1.5, label=f"Peak: {max(lp_f1s):.4f}")
    axes[0,1].set_xlabel("Pretrain Epoch"); axes[0,1].set_ylabel("Val Macro-F1")
    axes[0,1].set_title("Representation Quality (LP Monitor)\nPrimary convergence criterion")
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# EMA schedule
axes[1,0].plot(epochs, taus, color="#81c784", lw=2)
axes[1,0].axhline(EMA_TAU_START, color="#ccc", ls=":", lw=1, label=f"τ start = {EMA_TAU_START}")
axes[1,0].axhline(EMA_TAU_END,   color="#ccc", ls=":", lw=1, label=f"τ end   = {EMA_TAU_END}")
axes[1,0].set_xlabel("Epoch"); axes[1,0].set_ylabel("EMA Momentum τ")
axes[1,0].set_title("EMA Schedule (Cosine Warmup)"); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_ylim(EMA_TAU_START * 0.9995, 1.0005)

# LR schedule
axes[1,1].plot(epochs, lrs, color="#ce93d8", lw=2)
axes[1,1].set_xlabel("Epoch"); axes[1,1].set_ylabel("Learning Rate")
axes[1,1].set_title("Learning Rate Schedule (Warmup + Cosine)")
axes[1,1].set_yscale("log"); axes[1,1].grid(True, alpha=0.3)

plt.suptitle("Leaf-JEPA Pretraining Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_training_analysis.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✅ Training analysis figure saved")


✅ Training analysis figure saved


## Convergence Detection

In [3]:
# Detects the epoch where loss and LP F1 stopped meaningfully improving.

window = 10
deltas = [abs(rolling[i] - rolling[max(0, i-window)]) for i in range(len(rolling))]

convergence_loss_epoch = None
for i in range(window, len(deltas)):
    if deltas[i] < 0.001:
        convergence_loss_epoch = epochs[i]
        break

convergence_lp_epoch = None
if len(lp_f1s) >= 2:
    for i in range(1, len(lp_f1s)):
        improvement = lp_f1s[i] - max(lp_f1s[:i])
        if improvement < 0.005:
            convergence_lp_epoch = lp_epochs[i]
            break

print("Convergence analysis:")
print(f"  Loss converged (~delta < 0.001):  epoch {convergence_loss_epoch or 'not yet'}")
print(f"  LP F1 plateaued (< 0.005 gain):   epoch {convergence_lp_epoch  or 'not yet'}")
print()
print("Dissertation write-up:")
print(f"  'Training was stopped at epoch {epochs[-1]}. The pretraining loss")
print(f"   exhibited stable convergence from epoch {convergence_loss_epoch or '?'} onward,")
print(f"   with the linear probe validation macro-F1 reaching its peak of {max(lp_f1s) if lp_f1s else 0:.4f}")
print(f"   at epoch {lp_epochs[lp_f1s.index(max(lp_f1s))] if lp_f1s else '?'}.'")


Convergence analysis:
  Loss converged (~delta < 0.001):  epoch 49
  LP F1 plateaued (< 0.005 gain):   epoch 50

Dissertation write-up:
  'Training was stopped at epoch 125. The pretraining loss
   exhibited stable convergence from epoch 49 onward,
   with the linear probe validation macro-F1 reaching its peak of 0.8848
   at epoch 25.'


## Loss delta & Gradient Norm Analysis

In [4]:
grad_norms = [h.get("grad_norm", 0) for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs[window:], deltas[window:], color="#ef9a9a", lw=1.5)
axes[0].axhline(0.001, color="#ef5350", ls="--", lw=1.5, label="Convergence threshold (0.001)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel(f"|Δ loss| over {window} epochs")
axes[0].set_title("Loss Change Rate (Convergence Detection)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, grad_norms, color="#80cbc4", lw=1.5, alpha=0.7)
# Rolling avg
rolling_gn = [np.mean(grad_norms[max(0,i-window+1):i+1]) for i in range(len(grad_norms))]
axes[1].plot(epochs, rolling_gn, color="#00897b", lw=2, label="Rolling avg")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Gradient Norm")
axes[1].set_title("Gradient Norm (Stability Check)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Training Stability Analysis", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_convergence_analysis.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✅ Convergence analysis figure saved")
print("\n✅ S4_4 complete.")


✅ Convergence analysis figure saved

✅ S4_4 complete.
